# Customer Churn Prediction

Predicts which telecom customers are likely to cancel (churn), so a business can act before losing them.


In [1]:

import io, urllib.request
import pandas as pd, numpy as np
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
try:
    d = pd.read_csv(io.BytesIO(urllib.request.urlopen(req).read()))
    d["TotalCharges"] = pd.to_numeric(d["TotalCharges"], errors="coerce")
    d = d.dropna().drop(columns=["customerID"])
    X = pd.get_dummies(d.drop(columns=["Churn"]), drop_first=True)
    print("Real Telco dataset:", d.shape)
except Exception as e:
    print("Download failed, using a clearly-labelled synthetic dataset instead:", e)
    from sklearn.datasets import make_classification
    Xs, ys = make_classification(n_samples=5000, n_features=10, n_informative=7, weights=[0.73, 0.27], random_state=42)
    cols = ["tenure_months","monthly_charges","support_tickets","late_payments","logins_per_month","contract_score","paperless_billing","streaming_usage","data_usage_gb","household_size"]
    X = pd.DataFrame(Xs, columns=cols); d = X.copy(); d["Churn"] = np.where(ys == 1, "Yes", "No")
y = (d["Churn"] == "Yes").astype(int)
print("Churn rate:", round(float(y.mean()), 3))


Real Telco dataset: (7032, 20)
Churn rate: 0.266


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
models = {"Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42)}
for name, m in models.items():
    m.fit(X_train, y_train); print(name, "accuracy:", round(m.score(X_test, y_test), 4))
best_name = max(models, key=lambda n: models[n].score(X_test, y_test))
best_model = models[best_name]
print("Best:", best_name)


Logistic Regression accuracy: 0.8077


Random Forest accuracy: 0.7878
Best: Logistic Regression


In [3]:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pred = best_model.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"{best_name} accuracy: {acc:.4f}")
print(classification_report(y_test, pred))
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i,j] > cm.max()/2 else "black")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=120); plt.show()


Logistic Regression accuracy: 0.8077
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1291
           1       0.66      0.58      0.62       467

    accuracy                           0.81      1758
   macro avg       0.76      0.73      0.74      1758
weighted avg       0.80      0.81      0.80      1758



## Result

See README.md for the verified accuracy and which dataset variant was used.
